In [1]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/susmitmahato/codess/core/kitnet.py
/kaggle/input/datasets/susmitmahato/codess/core/_scipy_shim.py
/kaggle/input/datasets/susmitmahato/codess/core/feature_mapper.py
/kaggle/input/datasets/susmitmahato/codess/core/kitsune.py
/kaggle/input/datasets/susmitmahato/codess/core/feature_extractor.py
/kaggle/input/datasets/susmitmahato/codess/core/inc_stat.py
/kaggle/input/datasets/susmitmahato/codess/phase2/reader.py
/kaggle/input/datasets/susmitmahato/codess/phase2/find_attack_ips.py
/kaggle/input/datasets/susmitmahato/codess/phase2/README.md
/kaggle/input/datasets/susmitmahato/codess/phase2/relabel_features.py
/kaggle/input/datasets/susmitmahato/codess/phase2/combine_csvs.py
/kaggle/input/datasets/susmitmahato/codess/phase2/find_attack_pcaps.py
/kaggle/input/datasets/susmitmahato/codess/phase2/pipeline.py
/kaggle/input/datasets/susmitmahato/codess/phase2/__init__.py
/kaggle/input/datasets/susmitmahato/codess/phase2/csv_baseline/run_csv_baseline.py
/kaggle/input/datasets

In [1]:
import sys

CODE_ROOT    = '/kaggle/input/datasets/susmitmahato/codess'
COMBINED_DIR = '/kaggle/input/datasets/susmitmahato/combined'
OUTPUT_DIR   = '/kaggle/working/results_phase2/full'

sys.path.insert(0, CODE_ROOT)

In [3]:
import shutil, pathlib

# Copy pipeline to writable location and patch it
src = pathlib.Path(CODE_ROOT) / 'phase2/pipeline.py'
dst = pathlib.Path('/kaggle/working/pipeline_patched.py')
shutil.copy(src, dst)

txt = dst.read_text()

# Replace slow loop with fast numpy version
old = """    feat_matrix = df[feat_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    feat_matrix = feat_matrix.replace([np.inf, -np.inf], 0.0)

    for idx in range(len(df)):
        feat_vec    = feat_matrix.iloc[idx].values.astype(np.float32)
        bin_label   = int(df.iloc[idx]["binary_label"])
        attack_type = str(df.iloc[idx]["attack_type"])"""

new = """    feat_matrix = df[feat_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    feat_matrix = feat_matrix.replace([np.inf, -np.inf], 0.0)
    feat_array  = feat_matrix.values.astype(np.float32)
    label_array = df["binary_label"].values.astype(int)
    atype_array = df["attack_type"].values
    del feat_matrix

    for idx in range(len(df)):
        feat_vec    = feat_array[idx]
        bin_label   = int(label_array[idx])
        attack_type = str(atype_array[idx])"""

dst.write_text(txt.replace(old, new))
print("Patched!" if old in txt else "WARNING: pattern not found — check pipeline.py version")

Patched!


In [4]:
import importlib, sys
from pathlib import Path

# Load patched pipeline
spec = importlib.util.spec_from_file_location(
    "phase2.pipeline", "/kaggle/working/pipeline_patched.py"
)
mod = importlib.util.module_from_spec(spec)
sys.modules["phase2.pipeline"] = mod
spec.loader.exec_module(mod)

run_experiment = mod.run_experiment
print("Imported run_experiment successfully")

Imported run_experiment successfully


In [5]:
run_experiment(
    data_dir       = Path(COMBINED_DIR),
    output_dir     = Path(OUTPUT_DIR) / 'traditional',
    csv_suffix     = 'kitsune_115',
    feature_source = 'kitsune_network_115',
)

INFO Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO Using categorical un

[{'dataset': 'friday',
  'n_total': 1109491,
  'n_benign': 1050784,
  'n_malicious': 58707,
  'attack_rate': 0.052913,
  'AUC': 0.525194,
  'AUPRC': 0.05406,
  'EER': 0.478648,
  'TPR_at_FPR_0': 0.0,
  'FNR_at_FPR_0': 1.0,
  'threshold_FPR_0': inf,
  'TPR_at_FPR_0001': 0.0,
  'FNR_at_FPR_0001': 1.0,
  'threshold_FPR_0001': inf,
  'threshold_opt': 0.084426,
  'F1_optimal': 0.106061,
  'Precision_opt': 0.062869,
  'Recall_opt': 0.338869,
  'TP': 19894,
  'FP': 296542,
  'FN': 38813,
  'TN': 754242,
  'mean_score_benign': 0.390375,
  'mean_score_attack': 0.230376,
  'median_score_benign': 0.043357,
  'median_score_attack': 0.0495,
  'std_score_benign': 1.29812,
  'std_score_attack': 0.54895,
  'max_score': 22.387335,
  'min_score': 0.010001,
  'runtime_sec': 785.5532,
  'rows_per_sec': 1412.3691,
  'feature_source': 'kitsune_network_115',
  'n_features': 115,
  'total_rows': 1134491,
  'fm_grace': 5000,
  'ad_grace': 20000,
  'warmup_rows': 25000,
  'max_cluster_size': 10,
  'n_clusters':

In [6]:
run_experiment(
    data_dir       = Path(COMBINED_DIR),
    output_dir     = Path(OUTPUT_DIR) / 'http_only',
    csv_suffix     = 'http_85',
    feature_source = 'http_afterimage_85',
)

INFO Experiment 'http_afterimage_85' — 2 day(s) to process
INFO ======================================================================
INFO Experiment: http_afterimage_85  |  Day: friday
INFO feature_width=85  total_rows=1134491
INFO benign_prefix=138543  fm_grace=5000  ad_grace=20000
INFO Fitting FeatureMapper on 5000 rows…
INFO FeatureMapper ready — k=9 clusters
INFO Row 50000 / 1134491
INFO Row 100000 / 1134491
INFO Row 150000 / 1134491
INFO Row 200000 / 1134491
INFO Row 250000 / 1134491
INFO Row 300000 / 1134491
INFO Row 350000 / 1134491
INFO Row 400000 / 1134491
INFO Row 450000 / 1134491
INFO Row 500000 / 1134491
INFO Row 550000 / 1134491
INFO Row 600000 / 1134491
INFO Row 650000 / 1134491
INFO Row 700000 / 1134491
INFO Row 750000 / 1134491
INFO Row 800000 / 1134491
INFO Row 850000 / 1134491
INFO Row 900000 / 1134491
INFO Row 950000 / 1134491
INFO Row 1000000 / 1134491
INFO Row 1050000 / 1134491
INFO Row 1100000 / 1134491
INFO ------------------------------------------------------

[{'dataset': 'friday',
  'n_total': 1109491,
  'n_benign': 1050784,
  'n_malicious': 58707,
  'attack_rate': 0.052913,
  'AUC': 0.518869,
  'AUPRC': 0.05576,
  'EER': 0.48183,
  'TPR_at_FPR_0': 0.0,
  'FNR_at_FPR_0': 1.0,
  'threshold_FPR_0': inf,
  'TPR_at_FPR_0001': 0.001874,
  'FNR_at_FPR_0001': 0.998126,
  'threshold_FPR_0001': 3.227425,
  'threshold_opt': 0.019575,
  'F1_optimal': 0.103451,
  'Precision_opt': 0.059083,
  'Recall_opt': 0.415402,
  'TP': 24387,
  'FP': 388374,
  'FN': 34320,
  'TN': 662410,
  'mean_score_benign': 0.047212,
  'mean_score_attack': 0.052612,
  'median_score_benign': 0.015312,
  'median_score_attack': 0.01569,
  'std_score_benign': 0.215118,
  'std_score_attack': 0.250791,
  'max_score': 13.78853,
  'min_score': 0.009236,
  'runtime_sec': 463.5728,
  'rows_per_sec': 2393.3479,
  'feature_source': 'http_afterimage_85',
  'n_features': 85,
  'total_rows': 1134491,
  'fm_grace': 5000,
  'ad_grace': 20000,
  'warmup_rows': 25000,
  'max_cluster_size': 10,
 

In [7]:
run_experiment(
    data_dir       = Path(COMBINED_DIR),
    output_dir     = Path(OUTPUT_DIR) / 'hybrid',
    csv_suffix     = 'hybrid_200',
    feature_source = 'hybrid_network_http_200',
)

INFO Experiment 'hybrid_network_http_200' — 2 day(s) to process
INFO ======================================================================
INFO Experiment: hybrid_network_http_200  |  Day: friday
INFO feature_width=200  total_rows=1134491
INFO benign_prefix=138543  fm_grace=5000  ad_grace=20000
INFO Fitting FeatureMapper on 5000 rows…
INFO FeatureMapper ready — k=20 clusters
INFO Row 50000 / 1134491
INFO Row 100000 / 1134491
INFO Row 150000 / 1134491
INFO Row 200000 / 1134491
INFO Row 250000 / 1134491
INFO Row 300000 / 1134491
INFO Row 350000 / 1134491
INFO Row 400000 / 1134491
INFO Row 450000 / 1134491
INFO Row 500000 / 1134491
INFO Row 550000 / 1134491
INFO Row 600000 / 1134491
INFO Row 650000 / 1134491
INFO Row 700000 / 1134491
INFO Row 750000 / 1134491
INFO Row 800000 / 1134491
INFO Row 850000 / 1134491
INFO Row 900000 / 1134491
INFO Row 950000 / 1134491
INFO Row 1000000 / 1134491
INFO Row 1050000 / 1134491
INFO Row 1100000 / 1134491
INFO ------------------------------------------

[{'dataset': 'friday',
  'n_total': 1109491,
  'n_benign': 1050784,
  'n_malicious': 58707,
  'attack_rate': 0.052913,
  'AUC': 0.516864,
  'AUPRC': 0.052837,
  'EER': 0.483636,
  'TPR_at_FPR_0': 0.0,
  'FNR_at_FPR_0': 1.0,
  'threshold_FPR_0': inf,
  'TPR_at_FPR_0001': 0.0,
  'FNR_at_FPR_0001': 1.0,
  'threshold_FPR_0001': inf,
  'threshold_opt': 0.090767,
  'F1_optimal': 0.103115,
  'Precision_opt': 0.058839,
  'Recall_opt': 0.416611,
  'TP': 24458,
  'FP': 391216,
  'FN': 34249,
  'TN': 659568,
  'mean_score_benign': 0.402511,
  'mean_score_attack': 0.226776,
  'median_score_benign': 0.065224,
  'median_score_attack': 0.071933,
  'std_score_benign': 1.377297,
  'std_score_attack': 0.577017,
  'max_score': 22.305397,
  'min_score': 0.005558,
  'runtime_sec': 961.6425,
  'rows_per_sec': 1153.7458,
  'feature_source': 'hybrid_network_http_200',
  'n_features': 200,
  'total_rows': 1134491,
  'fm_grace': 5000,
  'ad_grace': 20000,
  'warmup_rows': 25000,
  'max_cluster_size': 10,
  'n_c

In [8]:
import json, pandas as pd

rows = []
for exp in ['traditional', 'http_only', 'hybrid']:
    for day in ['friday', 'thursday']:
        f = Path(OUTPUT_DIR) / exp / day / 'metrics.json'
        if f.exists():
            m = json.load(open(f))
            rows.append({'experiment': exp, 'day': day,
                         'AUC': m['AUC'], 'EER': m['EER'],
                         'n_attacks': m['n_malicious']})

pd.DataFrame(rows)

,experiment,day,AUC,EER,n_attacks
0,traditional,friday,0.525194,0.478648,58707
1,traditional,thursday,0.522342,0.477799,42836
2,http_only,friday,0.518869,0.481830,58707
3,http_only,thursday,0.539029,0.468001,42836
4,hybrid,friday,0.516864,0.483636,58707
5,hybrid,thursday,0.521132,0.477389,42836


In [9]:


import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS = Path("/kaggle/working/results_phase2/full")
PLOTS   = Path("/kaggle/working/extra_graphs")
PLOTS.mkdir(parents=True, exist_ok=True)

METHODS = {
    "Traditional 115": "traditional",
    "HTTP-only 85":    "http_only",
    "Hybrid 200":      "hybrid",
}
DAYS    = ["friday", "thursday"]
DAY_CAP = {"friday": "Friday", "thursday": "Thursday"}

METHOD_COLORS = {
    "Traditional 115": "#2196F3",
    "HTTP-only 85":    "#E91E63",
    "Hybrid 200":      "#4CAF50",
}
LABEL_COLORS = {0: "#2196F3", 1: "#E53935"}

def load_scores(method_key: str, day: str, sample: int = None) -> pd.DataFrame:
    path = RESULTS / method_key / day / "scores.csv"
    df = pd.read_csv(path)
    if sample and len(df) > sample:
        df = df.sample(n=sample, random_state=42).reset_index(drop=True)
    return df

def load_metrics(method_key: str, day: str) -> dict:
    path = RESULTS / method_key / day / "metrics.json"
    with open(path) as f:
        return json.load(f)

def load_curve(fname: str, method_key: str, day: str, step: int = 500) -> pd.DataFrame:
    path = RESULTS / method_key / day / fname
    df = pd.read_csv(path)
    return df.iloc[::step].reset_index(drop=True)

def load_attack_breakdown(method_key: str) -> pd.DataFrame:
    path = RESULTS / method_key / "attack_breakdown_combined.csv"
    return pd.read_csv(path)

def gaussian_kde_numpy(data: np.ndarray, x_grid: np.ndarray,
                       bw_factor: float = 1.0) -> np.ndarray:
    """Simple Gaussian KDE using numpy (no scipy required)."""
    n = len(data)
    bw = bw_factor * 1.06 * np.std(data) * n ** (-1 / 5)
    diff = x_grid[:, None] - data[None, :]           # (grid, n)
    kernel = np.exp(-0.5 * (diff / bw) ** 2) / (bw * np.sqrt(2 * np.pi))
    return kernel.mean(axis=1)

def rolling_mean_np(arr: np.ndarray, window: int) -> np.ndarray:
    """Fast rolling mean via pandas."""
    return pd.Series(arr).rolling(window, center=True, min_periods=1).mean().values

def save(fig, name: str):
    fp = PLOTS / name
    fig.savefig(fp, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved → {fp}")

print("Setup complete. Output folder:", PLOTS)
print("Methods:", list(METHODS.keys()))



fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle("Anomaly Score Distributions: Benign vs Attack\n(KDE estimate, 100 K sample)",
             fontsize=15, fontweight="bold", y=1.01)

for row_i, (label, mkey) in enumerate(METHODS.items()):
    for col_i, day in enumerate(DAYS):
        ax = axes[row_i][col_i]
        df = load_scores(mkey, day, sample=100_000)

        clip_val = float(np.percentile(df["score"].values, 99.5))
        benign   = df.loc[df["label"] == 0, "score"].values
        attacks  = df.loc[df["label"] == 1, "score"].values
        benign   = benign[benign <= clip_val]
        attacks  = attacks[attacks <= clip_val]

        x = np.linspace(0, clip_val, 400)
        for scores, color, name in [
            (benign,  LABEL_COLORS[0], "Benign"),
            (attacks, LABEL_COLORS[1], "Attack"),
        ]:
            if len(scores) > 30:
                # subsample for KDE speed
                if len(scores) > 20_000:
                    scores = np.random.default_rng(0).choice(scores, 20_000,
                                                             replace=False)
                kde_vals = gaussian_kde_numpy(scores, x)
                ax.plot(x, kde_vals, color=color, lw=2, label=name)
                ax.fill_between(x, kde_vals, alpha=0.15, color=color)

        ax.set_title(f"{label} — {DAY_CAP[day]}", fontsize=11, fontweight="bold")
        ax.set_xlabel("KitNET Anomaly Score")
        ax.set_ylabel("Density")
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
save(fig, "score_distributions_kde.png")




fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Precision-Recall Curves — All Methods",
             fontsize=14, fontweight="bold")

for col_i, day in enumerate(DAYS):
    ax = axes[col_i]
    for label, mkey in METHODS.items():
        pr = load_curve("pr_curve.csv", mkey, day, step=1000)
        m  = load_metrics(mkey, day)
        baseline = m["attack_rate"]
        ax.plot(pr["recall"], pr["precision"],
                color=METHOD_COLORS[label], lw=2,
                label=f"{label}  (AUPRC={m['AUPRC']:.4f})")

    ax.axhline(baseline, color="gray", linestyle="--", lw=1.2,
               label=f"Random baseline ({baseline:.3f})")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Recall", fontsize=11)
    ax.set_ylabel("Precision", fontsize=11)
    ax.set_title(DAY_CAP[day], fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
save(fig, "pr_curves_overlay.png")




fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Anomaly Score Distribution by Traffic Type\n"
             "(box=IQR, whiskers=1.5×IQR, outliers hidden, 50K sample)",
             fontsize=13, fontweight="bold", y=1.01)

ATYPES      = ["Benign", "Brute Force -Web", "SQL Injection"]
ATYPE_SHORT = {"Benign": "Benign", "Brute Force -Web": "BF-Web",
               "SQL Injection": "SQL-Inj"}
ATYPE_COLOR = {"Benign": "#2196F3", "Brute Force -Web": "#E53935",
               "SQL Injection": "#FF9800"}

for col_i, (label, mkey) in enumerate(METHODS.items()):
    for row_i, day in enumerate(DAYS):
        ax = axes[row_i][col_i]
        df = load_scores(mkey, day, sample=50_000)
        clip_val = float(np.percentile(df["score"].values, 99.5))
        df["score"] = df["score"].clip(upper=clip_val)

        groups, xticks, colors = [], [], []
        for at in ATYPES:
            grp = df.loc[df["attack_type"] == at, "score"].dropna().values
            if len(grp) > 0:
                groups.append(grp)
                xticks.append(ATYPE_SHORT[at])
                colors.append(ATYPE_COLOR[at])

        if groups:
            bp = ax.boxplot(groups, patch_artist=True, notch=False,
                            showfliers=False, widths=0.5)
            for patch, color in zip(bp["boxes"], colors):
                patch.set_facecolor(color); patch.set_alpha(0.6)

        ax.set_xticks(range(1, len(groups) + 1))
        ax.set_xticklabels(xticks, fontsize=9)
        ax.set_title(f"{label}\n{DAY_CAP[day]}", fontsize=10, fontweight="bold")
        ax.set_ylabel("Anomaly Score")
        ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
save(fig, "score_boxplots_by_attack_type.png")



WINDOW = 2000   # rows

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Anomaly Score Over Row Stream (temporal proxy)\n"
             f"Smoothed with window={WINDOW:,}",
             fontsize=13, fontweight="bold", y=1.01)

for col_i, (label, mkey) in enumerate(METHODS.items()):
    for row_i, day in enumerate(DAYS):
        ax = axes[row_i][col_i]
        path = RESULTS / mkey / day / "scores.csv"
        df = pd.read_csv(path).sort_values("row_index").reset_index(drop=True)
        clip_val = float(np.percentile(df["score"].values, 99.5))
        df["score"] = df["score"].clip(upper=clip_val)

        smooth = rolling_mean_np(df["score"].values, WINDOW)
        idx    = np.arange(len(df))

        # Shade attack regions
        attack_mask = (df["label"].values == 1)
        # group contiguous attack blocks for efficient shading
        in_block = False
        start_i  = 0
        for i, is_atk in enumerate(attack_mask):
            if is_atk and not in_block:
                start_i  = i; in_block = True
            elif not is_atk and in_block:
                ax.axvspan(start_i, i, color="#FFCDD2", alpha=0.3)
                in_block = False
        if in_block:
            ax.axvspan(start_i, len(attack_mask), color="#FFCDD2", alpha=0.3)

        ax.plot(idx, smooth, color=METHOD_COLORS[label], lw=1.2,
                label="Smoothed score")

        # dummy patch for legend
        from matplotlib.patches import Patch
        legend_elements = [
            plt.Line2D([0], [0], color=METHOD_COLORS[label], lw=2,
                       label="Smoothed score"),
            Patch(facecolor="#FFCDD2", alpha=0.5, label="Attack region"),
        ]
        ax.legend(handles=legend_elements, fontsize=8, loc="upper right")
        ax.set_title(f"{label} — {DAY_CAP[day]}", fontsize=10, fontweight="bold")
        ax.set_xlabel("Row index (stream order)")
        ax.set_ylabel("Anomaly Score")
        ax.grid(alpha=0.2)

plt.tight_layout()
save(fig, "temporal_score_plot.png")




fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Confusion Matrices at Optimal Threshold",
             fontsize=14, fontweight="bold", y=1.02)

for col_i, (label, mkey) in enumerate(METHODS.items()):
    for row_i, day in enumerate(DAYS):
        ax = axes[row_i][col_i]
        m  = load_metrics(mkey, day)
        cm = np.array([[m["TN"], m["FP"]],
                       [m["FN"], m["TP"]]], dtype=np.int64)
        total  = cm.sum()
        cm_pct = cm / total * 100

        im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=100)

        for i in range(2):
            for j in range(2):
                color = "white" if cm_pct[i, j] > 50 else "black"
                ax.text(j, i,
                        f"{cm[i, j]:,}\n({cm_pct[i, j]:.1f}%)",
                        ha="center", va="center",
                        fontsize=9, color=color, fontweight="bold")

        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(["Pred Benign", "Pred Attack"])
        ax.set_yticklabels(["True Benign", "True Attack"])
        ax.set_title(
            f"{label} — {DAY_CAP[day]}\n"
            f"F1={m['F1_optimal']:.4f}  thr={m['threshold_opt']:.4f}",
            fontsize=9, fontweight="bold"
        )
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label("% of total")

plt.tight_layout()
save(fig, "confusion_matrices.png")




fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Cumulative Distribution of Anomaly Scores\n(Benign vs Attack, 100K sample)",
             fontsize=13, fontweight="bold", y=1.01)

for col_i, (label, mkey) in enumerate(METHODS.items()):
    for row_i, day in enumerate(DAYS):
        ax = axes[row_i][col_i]
        df = load_scores(mkey, day, sample=100_000)
        clip_val = float(np.percentile(df["score"].values, 99.5))

        for lbl, color, name in [(0, LABEL_COLORS[0], "Benign"),
                                  (1, LABEL_COLORS[1], "Attack")]:
            vals = df.loc[df["label"] == lbl, "score"].values
            vals = np.sort(vals[vals <= clip_val])
            cdf  = np.arange(1, len(vals) + 1) / len(vals)
            ax.plot(vals, cdf, color=color, lw=2, label=name)

        # 95th percentile of benign = detection threshold
        benign_scores = df.loc[df["label"] == 0, "score"].values
        thr95 = float(np.percentile(benign_scores, 95))
        ax.axvline(thr95, color="orange", linestyle="--", lw=1.5,
                   label=f"p95 thr={thr95:.3f}")

        ax.set_title(f"{label} — {DAY_CAP[day]}", fontsize=10, fontweight="bold")
        ax.set_xlabel("Anomaly Score")
        ax.set_ylabel("Cumulative Probability")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
        ax.set_xlim(left=0, right=clip_val)

plt.tight_layout()
save(fig, "score_cdf.png")




rows = []
for label, mkey in METHODS.items():
    abd = load_attack_breakdown(mkey)
    for day in DAYS:
        d   = abd[abd["day"] == day]
        bf  = d[d["attack_type"] == "Brute Force -Web"]["detection_rate"].values
        sql = d[d["attack_type"] == "SQL Injection"]["detection_rate"].values
        m   = load_metrics(mkey, day)
        rows.append({
            "Method":      label,
            "Day":         DAY_CAP[day],
            "AUC":         round(m["AUC"],        4),
            "EER":         round(m["EER"],        4),
            "F1":          round(m["F1_optimal"], 4),
            "BF-Web Det%": round(float(bf[0])  * 100, 1) if len(bf)  else 0.0,
            "SQL Det%":    round(float(sql[0]) * 100, 1) if len(sql) else 0.0,
        })

sumdf = pd.DataFrame(rows)
sumdf["Exp"] = sumdf["Method"] + "\n" + sumdf["Day"]
sumdf = sumdf.set_index("Exp")

METRICS = ["AUC", "EER", "F1", "BF-Web Det%", "SQL Det%"]
heat_data = sumdf[METRICS]
mat = heat_data.values.astype(float)

# Normalise column-wise (0 → worst, 1 → best)
# EER: lower = better, so invert
mat_norm = np.zeros_like(mat)
for ci, metric in enumerate(METRICS):
    col = mat[:, ci]
    rng = col.max() - col.min()
    if rng < 1e-9:
        mat_norm[:, ci] = 0.5
    elif metric == "EER":
        mat_norm[:, ci] = 1 - (col - col.min()) / rng   # invert
    else:
        mat_norm[:, ci] = (col - col.min()) / rng

fig, ax = plt.subplots(figsize=(12, 7))
fig.suptitle("Metrics Summary Heatmap", fontsize=14, fontweight="bold")

im = ax.imshow(mat_norm.T, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)

ax.set_xticks(range(len(heat_data.index)))
ax.set_xticklabels(heat_data.index.tolist(), fontsize=9)
ax.set_yticks(range(len(METRICS)))
ax.set_yticklabels(METRICS, fontsize=11)

for i, metric in enumerate(METRICS):
    for j in range(len(heat_data.index)):
        val = heat_data.iloc[j][metric]
        nrm = mat_norm[j, i]
        color = "white" if nrm < 0.2 or nrm > 0.8 else "black"
        ax.text(j, i, str(val), ha="center", va="center",
                fontsize=9, fontweight="bold", color=color)

plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02).set_label(
    "Normalised (0=worst, 1=best per metric; EER inverted)")
ax.set_title("Green = better, Red = worse  (column-normalised)",
             fontsize=9, style="italic")
plt.tight_layout()
save(fig, "metrics_summary_heatmap.png")




gap_rows = []
for label, mkey in METHODS.items():
    for day in DAYS:
        m = load_metrics(mkey, day)
        gap = m["mean_score_attack"] - m["mean_score_benign"]
        gap_rows.append({
            "Method":      label,
            "Day":         DAY_CAP[day],
            "Gap":         round(gap, 6),
            "mean_benign": round(m["mean_score_benign"], 4),
            "mean_attack": round(m["mean_score_attack"], 4),
        })

gdf = pd.DataFrame(gap_rows)

fig, ax = plt.subplots(figsize=(10, 6))
mnames = list(METHODS.keys())
x = np.arange(len(mnames))
width = 0.35
day_colors = ["#5C6BC0", "#EF5350"]

for i, day in enumerate(DAYS):
    vals = [
        gdf[(gdf["Method"] == lbl) & (gdf["Day"] == DAY_CAP[day])]["Gap"].values[0]
        for lbl in mnames
    ]
    bars = ax.bar(x + i * width - width / 2, vals, width,
                  label=DAY_CAP[day], color=day_colors[i],
                  alpha=0.85, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, vals):
        va   = "bottom" if val >= 0 else "top"
        ypos = val + 0.0005 if val >= 0 else val - 0.0005
        ax.text(bar.get_x() + bar.get_width() / 2, ypos,
                f"{val:+.4f}", ha="center", va=va,
                fontsize=8, fontweight="bold")

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(mnames, fontsize=10)
ax.set_ylabel("Score Gap  (mean_attack − mean_benign)", fontsize=11)
ax.set_title("Score Separation: Attack vs Benign Mean Score\n"
             "Positive = attacks score higher (correct);  "
             "Negative = inverted signal (Traditional/Hybrid only)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save(fig, "score_gap_analysis.png")




print("\nAll graphs saved to:", PLOTS)
for f in sorted(PLOTS.glob("*.png")):
    print(" ", f.name)


Setup complete. Output folder: /kaggle/working/extra_graphs
Methods: ['Traditional 115', 'HTTP-only 85', 'Hybrid 200']
Saved → /kaggle/working/extra_graphs/score_distributions_kde.png
Saved → /kaggle/working/extra_graphs/pr_curves_overlay.png
Saved → /kaggle/working/extra_graphs/score_boxplots_by_attack_type.png
Saved → /kaggle/working/extra_graphs/temporal_score_plot.png
Saved → /kaggle/working/extra_graphs/confusion_matrices.png
Saved → /kaggle/working/extra_graphs/score_cdf.png
Saved → /kaggle/working/extra_graphs/metrics_summary_heatmap.png
Saved → /kaggle/working/extra_graphs/score_gap_analysis.png

All graphs saved to: /kaggle/working/extra_graphs
  confusion_matrices.png
  metrics_summary_heatmap.png
  pr_curves_overlay.png
  score_boxplots_by_attack_type.png
  score_cdf.png
  score_distributions_kde.png
  score_gap_analysis.png
  temporal_score_plot.png


In [10]:
!zip -r my_folders.zip /kaggle/working/results_phase2 /kaggle/working/extra_graphs

  adding: kaggle/working/results_phase2/ (stored 0%)
  adding: kaggle/working/results_phase2/full/ (stored 0%)
  adding: kaggle/working/results_phase2/full/traditional/ (stored 0%)
  adding: kaggle/working/results_phase2/full/traditional/summary_metrics.csv (deflated 52%)
  adding: kaggle/working/results_phase2/full/traditional/_plots/ (stored 0%)
  adding: kaggle/working/results_phase2/full/traditional/_plots/summary_runtime.png (deflated 30%)
  adding: kaggle/working/results_phase2/full/traditional/_plots/thursday_score_dist.png (deflated 24%)
  adding: kaggle/working/results_phase2/full/traditional/_plots/combined_roc.png (deflated 12%)
  adding: kaggle/working/results_phase2/full/traditional/_plots/summary_auc.png (deflated 31%)
  adding: kaggle/working/results_phase2/full/traditional/_plots/summary_metrics_sorted.csv (deflated 52%)
  adding: kaggle/working/results_phase2/full/traditional/_plots/friday_score_dist.png (deflated 24%)
  adding: kaggle/working/results_phase2/full/tradi

In [5]:
"""
Architecture changes vs Phase 2 HTTP-only
------------------------------------------
  1. HTTPFeatureMapper: max_cluster_size=5, min_clusters=15
     → forces ≥15 sub-autoencoders (Phase 2 got only 9)
  2. HTTPAutoencoder: Leaky ReLU hidden layer + EMA input normalisation
     → more stable gradients for sparse HTTP statistics
  3. HTTPKitNET: per-cluster RMSE z-score normalisation before output AE
     → equal contribution from all clusters regardless of size
  4. Warmup: fm_grace=min(8000,20%×benign), ad_grace=min(40000,75%×benign)
     → more warmup diversity, appropriate for HTTP transaction rate
"""


from __future__ import annotations

import json
import logging
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from scipy.cluster.hierarchy import fcluster, linkage
    from scipy.spatial.distance import squareform
    from scipy.stats import rankdata
    _SCIPY = True
except ImportError:
    _SCIPY = False

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
_log = logging.getLogger(__name__)


if not _SCIPY:
    _log.warning("scipy not found — using numpy fallback (results may differ slightly)")

    def squareform(X, checks=False):
        n = X.shape[0]
        out = []
        for i in range(n):
            for j in range(i + 1, n):
                out.append(X[i, j])
        return np.array(out)

    def linkage(condensed, method="complete"):
        # Minimal single/complete linkage via naive O(n^3) algorithm
        n = int(round((1 + np.sqrt(1 + 8 * len(condensed))) / 2))
        D = np.zeros((n, n))
        k = 0
        for i in range(n):
            for j in range(i + 1, n):
                D[i, j] = D[j, i] = condensed[k]; k += 1
        clusters = {i: [i] for i in range(n)}
        Z = []
        active = list(range(n))
        next_id = n
        while len(active) > 1:
            best = np.inf; bi = bj = -1
            for ii in range(len(active)):
                for jj in range(ii + 1, len(active)):
                    ci, cj = active[ii], active[jj]
                    if method == "complete":
                        d = max(D[a][b] for a in clusters[ci] for b in clusters[cj])
                    else:
                        d = min(D[a][b] for a in clusters[ci] for b in clusters[cj])
                    if d < best:
                        best = d; bi = ii; bj = jj
            ci, cj = active[bi], active[bj]
            clusters[next_id] = clusters[ci] + clusters[cj]
            Z.append([ci, cj, best, len(clusters[next_id])])
            active.pop(bj); active.pop(bi); active.append(next_id)
            next_id += 1
        return np.array(Z)

    def fcluster(Z, t, criterion="maxclust"):
        n = len(Z) + 1
        parent = {i: i for i in range(2 * n - 1)}
        def find(x):
            while parent[x] != x: x = parent[x]
            return x
        edges = sorted(enumerate(Z), key=lambda x: x[1][2])
        labels = list(range(n))
        # Simple: cut at t clusters
        union = {i: {i} for i in range(n)}
        for idx, row in enumerate(Z):
            a, b = int(row[0]), int(row[1])
            ra, rb = find(a), find(b)
            if ra != rb:
                union[ra] = union.get(ra, {ra}) | union.get(rb, {rb})
                parent[rb] = ra
        return np.array(labels) + 1


# 2.  HTTPFeatureMapper

class HTTPFeatureMapper:
    def __init__(self, n_features=85, max_cluster_size=5, min_clusters=15):
        self.n = n_features
        self.max_cluster_size = max_cluster_size
        self.min_clusters = min_clusters
        self._c    = np.zeros(self.n, dtype=np.float64)
        self._c_rs = np.zeros(self.n, dtype=np.float64)
        self._C    = np.zeros((self.n, self.n), dtype=np.float64)
        self._t    = 0
        self.mapping:   list[list[int]] | None = None
        self.is_fitted: bool = False

    def update(self, x):
        self._t += 1
        self._c += x
        mu = self._c / self._t
        r = x - mu
        self._c_rs += r ** 2
        self._C += np.outer(r, r)

    def fit(self):
        D = np.ones((self.n, self.n), dtype=np.float64)
        denom = np.sqrt(np.outer(self._c_rs, self._c_rs))
        nz = denom > 1e-12
        D[nz] = 1.0 - self._C[nz] / denom[nz]
        D = np.clip(D, 0.0, 1.0)
        np.fill_diagonal(D, 0.0)
        D = (D + D.T) / 2.0
        condensed = squareform(D, checks=False)
        Z = linkage(condensed, method="complete")
        self.mapping = self._cut_and_enforce(Z)
        self.is_fitted = True

    def _cut_and_enforce(self, Z):
        labels = fcluster(Z, t=self.max_cluster_size, criterion="maxclust")
        groups = {}
        for feat_idx, cid in enumerate(labels):
            groups.setdefault(int(cid), []).append(feat_idx)
        capped = []
        for grp in groups.values():
            for i in range(0, len(grp), self.max_cluster_size):
                capped.append(grp[i: i + self.max_cluster_size])
        # enforce min_clusters
        while len(capped) < self.min_clusters:
            splittable = [i for i, g in enumerate(capped) if len(g) > 1]
            if not splittable:
                break
            li = max(splittable, key=lambda i: len(capped[i]))
            lg = capped.pop(li)
            mid = len(lg) // 2
            capped.append(lg[:mid])
            capped.append(lg[mid:])
        return capped

    def transform(self, x):
        return [x[idx] for idx in self.mapping]

    @property
    def n_clusters(self):
        return len(self.mapping) if self.is_fitted else 0

    @property
    def cluster_sizes(self):
        return [len(g) for g in self.mapping] if self.is_fitted else []


# 3.  HTTPAutoencoder

class HTTPAutoencoder:
    def __init__(self, n_visible, n_hidden, lr=0.1, ema_momentum=0.99, leaky_alpha=0.01):
        self.n_visible    = n_visible
        self.n_hidden     = n_hidden
        self.lr           = lr
        self.ema_momentum = ema_momentum
        self.leaky_alpha  = leaky_alpha

        bound = 1.0 / max(n_visible, 1)
        rng   = np.random.default_rng()
        self.W_enc = rng.uniform(-bound, bound, (n_hidden, n_visible)).astype(np.float64)
        self.b_enc = np.zeros(n_hidden,  dtype=np.float64)
        self.W_dec = rng.uniform(-bound, bound, (n_visible, n_hidden)).astype(np.float64)
        self.b_dec = np.zeros(n_visible, dtype=np.float64)

        self._ema_mean = np.zeros(n_visible, dtype=np.float64)
        self._ema_var  = np.ones(n_visible,  dtype=np.float64)
        self._n_seen   = 0

    def _update_ema(self, x):
        self._n_seen += 1
        if self._n_seen == 1:
            self._ema_mean = x.copy()
        else:
            m = self.ema_momentum
            prev = self._ema_mean.copy()
            self._ema_mean = m * self._ema_mean + (1 - m) * x
            self._ema_var  = m * self._ema_var  + (1 - m) * (x - prev) ** 2

    def _normalise(self, x):
        return (x - self._ema_mean) / (np.sqrt(self._ema_var + 1e-8))

    def _leaky_relu(self, z):
        return np.where(z > 0, z, self.leaky_alpha * z)

    def _leaky_relu_grad(self, z):
        return np.where(z > 0, 1.0, self.leaky_alpha)

    @staticmethod
    def _sigmoid(z):
        z = np.clip(z, -500.0, 500.0)
        return np.where(z >= 0, 1.0/(1.0+np.exp(-z)), np.exp(z)/(1.0+np.exp(z)))

    @staticmethod
    def _sigmoid_grad(a):
        return a * (1.0 - a)

    @staticmethod
    def _rmse(a, b):
        return float(np.sqrt(np.mean((a - b) ** 2)))

    def _forward(self, x_norm):
        z_enc  = self.W_enc @ x_norm + self.b_enc
        hidden = self._leaky_relu(z_enc)
        output = self._sigmoid(self.W_dec @ hidden + self.b_dec)
        return z_enc, hidden, output

    def train(self, x):
        self._update_ema(x)
        x_norm = self._normalise(x)
        target = self._sigmoid(x_norm)
        z_enc, hidden, output = self._forward(x_norm)
        err = self._rmse(target, output)
        out_d = np.clip((output - target) * self._sigmoid_grad(output), -1.0, 1.0)
        hid_d = np.clip((self.W_dec.T @ out_d) * self._leaky_relu_grad(z_enc), -1.0, 1.0)
        self.W_dec -= self.lr * np.outer(out_d,  hidden)
        self.b_dec -= self.lr * out_d
        self.W_enc -= self.lr * np.outer(hid_d,  x_norm)
        self.b_enc -= self.lr * hid_d
        return err

    def execute(self, x):
        x_norm = self._normalise(x)
        target = self._sigmoid(x_norm)
        _, _, output = self._forward(x_norm)
        return self._rmse(target, output)


# 4.  HTTPKitNET

class HTTPKitNET:
    def __init__(self, cluster_sizes, beta=0.75, lr=0.1):
        self.k = len(cluster_sizes)
        self.ensemble = [
            HTTPAutoencoder(sz, max(1, int(np.ceil(beta * sz))), lr)
            for sz in cluster_sizes
        ]
        self.output_layer = HTTPAutoencoder(
            self.k, max(1, int(np.ceil(beta * self.k))), lr
        )
        self.phi = 0.0
        self._rmse_sum = np.zeros(self.k, dtype=np.float64)
        self._rmse_sq  = np.zeros(self.k, dtype=np.float64)
        self._rmse_n   = 0
        self._rmse_mean = np.zeros(self.k, dtype=np.float64)
        self._rmse_std  = np.ones(self.k,  dtype=np.float64)

    def _raw_errors(self, sub_instances, training):
        e = np.zeros(self.k, dtype=np.float64)
        for i, (ae, vi) in enumerate(zip(self.ensemble, sub_instances)):
            e[i] = ae.train(vi) if training else ae.execute(vi)
        return e

    def _update_stats(self, errors):
        self._rmse_n   += 1
        self._rmse_sum += errors
        self._rmse_sq  += errors ** 2
        n = self._rmse_n
        self._rmse_mean = self._rmse_sum / n
        var = np.maximum(self._rmse_sq / n - self._rmse_mean ** 2, 1e-12)
        self._rmse_std  = np.sqrt(var)

    def _normalise_errors(self, errors):
        if self._rmse_n == 0:
            return errors
        return (errors - self._rmse_mean) / (self._rmse_std + 1e-8)

    def train(self, sub_instances):
        errors = self._raw_errors(sub_instances, training=True)
        self._update_stats(errors)
        norm   = self._normalise_errors(errors)
        score  = self.output_layer.train(norm)
        self.phi = max(self.phi, score)
        return score

    def execute(self, sub_instances):
        errors = self._raw_errors(sub_instances, training=False)
        norm   = self._normalise_errors(errors)
        return self.output_layer.execute(norm)


# 5.  Warmup policy

def compute_http_warmup(benign_prefix):
    fm = min(8_000,  int(0.20 * benign_prefix))
    ad = min(40_000, int(0.75 * benign_prefix))
    if fm + ad > benign_prefix:
        ratio = benign_prefix / (fm + ad)
        fm = int(fm * ratio)
        ad = int(ad * ratio)
    return fm, ad


# 6.  Pipeline state machine

class HTTPPipeline:
    def __init__(self, fm_grace, ad_grace, n_features,
                 max_cluster_size=5, min_clusters=15, beta=0.75, lr=0.1):
        self.fm_grace = fm_grace
        self.ad_grace = ad_grace
        self.feat_mapper = HTTPFeatureMapper(n_features, max_cluster_size, min_clusters)
        self.detector    = None
        self._ready      = False
        self._rows_seen  = 0
        self.peak_train_score = 0.0
        self._beta = beta; self._lr = lr

    @property
    def warmup_length(self):
        return self.fm_grace + self.ad_grace

    def step(self, feat_vec):
        idx = self._rows_seen; self._rows_seen += 1
        if idx < self.fm_grace:
            self.feat_mapper.update(feat_vec)
            return None, "fm_train"
        if not self._ready:
            _log.info("Fitting HTTPFeatureMapper on %d rows …", self.fm_grace)
            self.feat_mapper.fit()
            self.detector = HTTPKitNET(self.feat_mapper.cluster_sizes, self._beta, self._lr)
            self._ready = True
            _log.info("k=%d clusters  sizes=%s",
                      self.feat_mapper.n_clusters, self.feat_mapper.cluster_sizes)
        sub = self.feat_mapper.transform(feat_vec)
        if idx < self.warmup_length:
            score = float(self.detector.train(sub))
            self.peak_train_score = max(self.peak_train_score, score)
            return None, "ad_train"
        return float(self.detector.execute(sub)), "exec"


# 7.  Metrics helpers  (fully vectorised — O(N log N), no Python loops)

def _sorted_cumulative(scores, labels):
    """
    Sort by score descending; return (sorted_labels, tp_cumsum, fp_cumsum,
    pos, neg, sorted_scores).  All subsequent metric functions share this.
    """
    order         = np.argsort(scores)[::-1]
    sorted_labels = labels[order].astype(np.int32)
    sorted_scores = scores[order]
    pos           = int(sorted_labels.sum())
    neg           = len(sorted_labels) - pos
    tp            = np.cumsum(sorted_labels)
    fp            = np.cumsum(1 - sorted_labels)
    return sorted_labels, tp, fp, pos, neg, sorted_scores


def _auc_roc(scores, labels):
    """Trapezoidal AUC-ROC.  O(N log N)."""
    _, tp, fp, pos, neg, _ = _sorted_cumulative(scores, labels)
    if pos == 0 or neg == 0:
        return 0.5
    tpr = np.concatenate([[0.0], tp / pos])
    fpr = np.concatenate([[0.0], fp / neg])
    return float(np.trapz(tpr, fpr))


def _auprc(scores, labels):
    """Area under the precision-recall curve.  O(N log N)."""
    _, tp, fp, pos, _, _ = _sorted_cumulative(scores, labels)
    if pos == 0:
        return 0.0
    prec = tp / (tp + fp)
    rec  = tp / pos
    return float(np.trapz(prec, rec))


def _eer(scores, labels):
    """Equal error rate.  O(N log N)."""
    _, tp, fp, pos, neg, _ = _sorted_cumulative(scores, labels)
    if pos == 0 or neg == 0:
        return 0.5
    fnr  = 1.0 - tp / pos
    fpr  = fp / neg
    idx  = int(np.argmin(np.abs(fnr - fpr)))
    return float((fnr[idx] + fpr[idx]) / 2.0)


def _optimal_f1(scores, labels):
    """
    Optimal F1 over all thresholds.  O(N log N) — fully vectorised.
    Previously O(N²) with a Python threshold loop, which hangs on 1M rows.
    """
    _, tp, fp, pos, neg, _ = _sorted_cumulative(scores, labels)
    if pos == 0:
        return 0.0, 0.0, 0.0, 0, 0, pos, neg

    fn   = pos - tp
    tn   = neg - fp
    prec = np.where((tp + fp) > 0, tp / (tp + fp), 0.0)
    rec  = np.where(pos > 0,       tp / pos,        0.0)
    denom = prec + rec
    f1   = np.where(denom > 0, 2.0 * prec * rec / denom, 0.0)

    idx = int(np.argmax(f1))
    return (float(f1[idx]), float(prec[idx]), float(rec[idx]),
            int(tp[idx]), int(fp[idx]), int(fn[idx]), int(tn[idx]))


def compute_metrics_local(scores, labels, day_name, runtime_sec, extra):
    auc   = _auc_roc(scores, labels)
    auprc = _auprc(scores, labels)
    eer   = _eer(scores, labels)
    f1, prec, rec, tp, fp, fn, tn = _optimal_f1(scores, labels)
    ben = scores[labels == 0]; att = scores[labels == 1]
    m = {
        "dataset": day_name, "AUC": round(auc, 6), "AUPRC": round(auprc, 6),
        "EER": round(eer, 6), "F1_optimal": round(f1, 6),
        "Recall_opt": round(rec, 6), "Precision_opt": round(prec, 6),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "n_total": len(labels), "n_benign": int((labels==0).sum()),
        "n_malicious": int((labels==1).sum()),
        "attack_rate": round(float((labels==1).mean()), 6),
        "mean_score_benign": round(float(ben.mean()) if len(ben) else 0.0, 6),
        "mean_score_attack": round(float(att.mean()) if len(att) else 0.0, 6),
        "runtime_sec": round(runtime_sec, 2),
    }
    m.update(extra)
    return m


def build_curve_frames_local(scores, labels):
    """Build ROC and PR curve DataFrames.  O(N log N), no Python loops."""
    _, tp, fp, pos, neg, sc_s = _sorted_cumulative(scores, labels)
    tpr  = tp / pos  if pos else np.zeros_like(tp, dtype=np.float64)
    fpr  = fp / neg  if neg else np.zeros_like(fp, dtype=np.float64)
    prec = np.where((tp + fp) > 0, tp / (tp + fp), 0.0)
    rec  = tp / pos  if pos else np.zeros_like(tp, dtype=np.float64)
    roc_df = pd.DataFrame({"fpr": fpr, "tpr": tpr, "threshold": sc_s})
    pr_df  = pd.DataFrame({"precision": prec, "recall": rec, "threshold": sc_s})
    return roc_df, pr_df


# 8.  Attack breakdown

def build_attack_breakdown(scored_df):
    benign_scores = scored_df.loc[scored_df["label"] == 0, "score"]
    threshold = float(np.percentile(benign_scores, 95)) if len(benign_scores) else 0.0
    rows = []
    for atype, grp in scored_df.groupby("attack_type"):
        total    = len(grp)
        detected = int((grp["score"] > threshold).sum())
        rows.append({
            "attack_type": atype, "total_rows": total,
            "attack_rows": int(grp["label"].sum()),
            "detected": detected, "missed": total - detected,
            "detection_rate": round(detected / total, 4) if total else 0.0,
            "threshold_p95": round(threshold, 6),
        })
    return pd.DataFrame(rows).sort_values("attack_type").reset_index(drop=True)


# 9.  Single-day evaluation

def evaluate_day(day_name, csv_path, output_dir,
                 max_cluster_size=5, min_clusters=15, beta=0.75, lr=0.1):
    _log.info("=" * 70)
    _log.info("Phase 3  |  Day: %s", day_name)

    CHUNK     = 100_000
    META_COLS = {"binary_label", "attack_type", "day", "timestamp", "label_source"}

    hdr       = pd.read_csv(csv_path, nrows=0)
    feat_cols = [c for c in hdr.columns if c not in META_COLS]
    n_feats   = len(feat_cols)
    read_cols = feat_cols + ["binary_label", "attack_type"]
    _log.info("feature_width=%d", n_feats)

    # Pass 1 — benign prefix scan
    benign_prefix = 0; total_rows = 0; found_attack = False
    for chunk in pd.read_csv(csv_path, usecols=["binary_label"],
                              chunksize=CHUNK, dtype={"binary_label": "int8"}):
        total_rows += len(chunk)
        if not found_attack:
            for lbl in chunk["binary_label"]:
                if int(lbl) == 1: found_attack = True; break
                benign_prefix += 1

    fm_grace, ad_grace = compute_http_warmup(benign_prefix)
    _log.info("benign_prefix=%d  fm=%d  ad=%d", benign_prefix, fm_grace, ad_grace)

    pipeline = HTTPPipeline(fm_grace, ad_grace, n_feats,
                            max_cluster_size, min_clusters, beta, lr)

    scored_rows: list[dict] = []
    t0 = time.time(); glob_idx = 0
    feat_dtype = {c: "float32" for c in feat_cols}

    # Pass 2 — stream through pipeline
    for chunk in pd.read_csv(csv_path, usecols=read_cols, chunksize=CHUNK,
                              dtype=feat_dtype, low_memory=False):
        fa = chunk[feat_cols].fillna(0.0).replace([np.inf, -np.inf], 0.0).values.astype(np.float32)
        la = chunk["binary_label"].values.astype(int)
        aa = chunk["attack_type"].values

        for i in range(len(chunk)):
            fv = fa[i]; bl = int(la[i]); at = str(aa[i])
            if pipeline._rows_seen < pipeline.warmup_length and bl == 1:
                scored_rows.append({"row_index": glob_idx, "day": day_name,
                                    "label": bl, "score": 0.0, "attack_type": at})
                glob_idx += 1; continue
            score, phase = pipeline.step(fv)
            if phase == "exec":
                scored_rows.append({"row_index": glob_idx, "day": day_name,
                                    "label": bl, "score": float(score), "attack_type": at})
            glob_idx += 1

        if glob_idx % 200_000 < CHUNK:
            _log.info("Row %d / %d  %.0f r/s", glob_idx, total_rows,
                      glob_idx / max(time.time()-t0, 1))

    elapsed = time.time() - t0
    if not scored_rows:
        return {"dataset": day_name, "error": "no_eval_rows"}

    scored_df = pd.DataFrame(scored_rows)
    scores = scored_df["score"].to_numpy(np.float64)
    labels = scored_df["label"].to_numpy(np.int32)

    metrics = compute_metrics_local(scores, labels, day_name, elapsed, {
        "feature_source":   "http_afterimage_85_phase3",
        "n_features":       n_feats, "total_rows": total_rows,
        "fm_grace":         fm_grace, "ad_grace": ad_grace,
        "warmup_rows":      pipeline.warmup_length,
        "max_cluster_size": max_cluster_size, "min_clusters": min_clusters,
        "n_clusters":       pipeline.feat_mapper.n_clusters,
        "cluster_sizes":    pipeline.feat_mapper.cluster_sizes,
        "peak_train_score": round(pipeline.peak_train_score, 6),
        "architecture":     "HTTPKitNET_phase3",
    })

    _log.info("[%s] AUC=%.4f  AUPRC=%.4f  EER=%.4f  F1=%.4f",
              day_name, metrics["AUC"], metrics["AUPRC"],
              metrics["EER"], metrics["F1_optimal"])

    out = output_dir / day_name
    out.mkdir(parents=True, exist_ok=True)
    scored_df.to_csv(out / "scores.csv", index=False)
    roc_df, pr_df = build_curve_frames_local(scores, labels)
    roc_df.to_csv(out / "roc_curve.csv", index=False)
    pr_df.to_csv(out / "pr_curve.csv",  index=False)
    with open(out / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    bd = build_attack_breakdown(scored_df)
    bd.to_csv(out / "attack_breakdown.csv", index=False)
    _log.info("Attack breakdown:\n%s", bd.to_string(index=False))
    return metrics


# Phase 2 vs Phase 3 comparison table


def build_comparison(phase2_results_dir: Path, phase3_results_dir: Path) -> pd.DataFrame:
    """
    Load Phase 2 HTTP-only metrics.json files and Phase 3 metrics.json files,
    side-by-side comparison of key metrics per day.
    """
    rows = []
    for day in ["thursday", "friday"]:
        p2_path = phase2_results_dir / "http_only" / day / "metrics.json"
        p3_path = phase3_results_dir / day / "metrics.json"

        p2 = json.loads(p2_path.read_text()) if p2_path.exists() else {}
        p3 = json.loads(p3_path.read_text()) if p3_path.exists() else {}

        for key in ["AUC", "AUPRC", "EER", "F1_optimal"]:
            rows.append({
                "day":    day,
                "metric": key,
                "phase2_http_only": round(p2.get(key, float("nan")), 4),
                "phase3_http_tuned": round(p3.get(key, float("nan")), 4),
                "delta": round(
                    p3.get(key, float("nan")) - p2.get(key, float("nan")), 4
                ),
            })

        # Detection rates
        for bd_day_name, bd_dir in [("phase2", p2_path.parent), ("phase3", p3_path.parent)]:
            bd_file = bd_dir / "attack_breakdown.csv"
            if bd_file.exists():
                bd = pd.read_csv(bd_file)
                for _, r in bd.iterrows():
                    if r["attack_type"] == "Benign":
                        continue
                    rows.append({
                        "day":    day,
                        "metric": f"DetRate_{r['attack_type']}",
                        "phase2_http_only":  round(bd.loc[bd["attack_type"]==r["attack_type"],
                                                    "detection_rate"].values[0], 4)
                                             if bd_day_name == "phase2" else float("nan"),
                        "phase3_http_tuned": round(r["detection_rate"], 4)
                                             if bd_day_name == "phase3" else float("nan"),
                        "delta": float("nan"),
                    })

    return pd.DataFrame(rows)


def main():
    # ── Paths ─────────────────────────────────────────────────────────────────
=    COMBINED_DIR   = Path("/kaggle/input/datasets/susmitmahato/combined")
    PHASE3_OUT     = Path("/kaggle/working/results_phase3")
    PHASE2_OUT     = Path("/kaggle/input/kitnet-phase2-data/results_phase2/full")

    # Fallback: try common alternative paths
    if not COMBINED_DIR.exists():
        candidates = list(Path("/kaggle/input").glob("*/combined"))
        if candidates:
            COMBINED_DIR = candidates[0]
            _log.info("Auto-detected COMBINED_DIR: %s", COMBINED_DIR)
        else:
            _log.error("Cannot find combined/ directory. Set COMBINED_DIR manually.")
            sys.exit(1)

    PHASE3_OUT.mkdir(parents=True, exist_ok=True)

    # ── Discover HTTP-85 CSVs 
    http_csvs = sorted(COMBINED_DIR.glob("*_http_85.csv"))
    if not http_csvs:
        _log.error("No *_http_85.csv files in %s", COMBINED_DIR)
        sys.exit(1)

    _log.info("Found %d HTTP-85 CSV(s): %s", len(http_csvs), [p.name for p in http_csvs])

    # ── Run Phase 3 
    all_metrics = []
    for csv_path in http_csvs:
        day_name = csv_path.stem.replace("_http_85", "")
        result   = evaluate_day(
            day_name         = day_name,
            csv_path         = csv_path,
            output_dir       = PHASE3_OUT,
            max_cluster_size = 5,
            min_clusters     = 15,
            beta             = 0.75,
            lr               = 0.1,
        )
        all_metrics.append(result)

    # ── Summary ───────────────────────────────────────────────────────────────
    summary_df = pd.DataFrame(all_metrics)
    summary_df.to_csv(PHASE3_OUT / "summary_metrics.csv", index=False)
    with open(PHASE3_OUT / "summary_metrics.json", "w") as f:
        json.dump(all_metrics, f, indent=2)

    # Combine attack breakdowns
    bds = []
    for csv_path in http_csvs:
        day_name = csv_path.stem.replace("_http_85", "")
        bd_file  = PHASE3_OUT / day_name / "attack_breakdown.csv"
        if bd_file.exists():
            bd = pd.read_csv(bd_file); bd.insert(0, "day", day_name)
            bds.append(bd)
    if bds:
        pd.concat(bds).to_csv(PHASE3_OUT / "attack_breakdown_combined.csv", index=False)

    # Phase 2 vs Phase 3 comparison
    if PHASE2_OUT.exists():
        try:
            comp_df = build_comparison(PHASE2_OUT, PHASE3_OUT)
            comp_df.to_csv(PHASE3_OUT / "phase2_vs_phase3_comparison.csv", index=False)
            _log.info("\nPhase 2 vs Phase 3 comparison:\n%s", comp_df.to_string(index=False))
        except Exception as exc:
            _log.warning("Comparison table failed (non-fatal): %s", exc)

    _log.info("\n" + "=" * 70)
    _log.info("Phase 3 complete.  Results in: %s", PHASE3_OUT)
    _log.info("Summary:")
    for m in all_metrics:
        _log.info(
            "  [%s]  AUC=%.4f  AUPRC=%.4f  EER=%.4f  F1=%.4f  k=%s",
            m.get("dataset", "?"), m.get("AUC", 0), m.get("AUPRC", 0),
            m.get("EER", 0), m.get("F1_optimal", 0), m.get("n_clusters", "?"),
        )


if __name__ == "__main__":
    main()

2026-04-24 11:40:08,369 [INFO] Found 2 HTTP-85 CSV(s): ['friday_http_85.csv', 'thursday_http_85.csv']
2026-04-24 11:40:08,370 [INFO] ======================================================================
2026-04-24 11:40:08,371 [INFO] Phase 3  |  Day: friday
2026-04-24 11:40:08,382 [INFO] feature_width=85
2026-04-24 11:40:13,538 [INFO] benign_prefix=138543  fm=8000  ad=40000
2026-04-24 11:40:15,277 [INFO] Fitting HTTPFeatureMapper on 8000 rows …
2026-04-24 11:40:15,281 [INFO] k=17 clusters  sizes=[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
2026-04-24 11:44:12,777 [INFO] Row 200000 / 1134491  836 r/s
2026-04-24 11:47:45,004 [INFO] Row 400000 / 1134491  886 r/s
2026-04-24 11:51:17,540 [INFO] Row 600000 / 1134491  904 r/s
2026-04-24 11:54:48,874 [INFO] Row 800000 / 1134491  914 r/s
2026-04-24 11:58:20,618 [INFO] Row 1000000 / 1134491  920 r/s


/tmp/ipykernel_55/3684373336.py:424: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipykernel_55/3684373336.py:434: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(prec, rec))


2026-04-24 12:00:46,274 [INFO] [friday] AUC=0.5291  AUPRC=0.0621  EER=0.4823  F1=0.1128


/tmp/ipykernel_55/3684373336.py:462: RuntimeWarning: invalid value encountered in divide
  f1   = np.where(denom > 0, 2.0 * prec * rec / denom, 0.0)


2026-04-24 12:01:01,090 [INFO] Attack breakdown:
     attack_type  total_rows  attack_rows  detected  missed  detection_rate  threshold_p95
          Benign     1027784            0     51390  976394           0.050       0.012516
Brute Force -Web       58397        58397      4495   53902           0.077       0.012516
   SQL Injection         310          310        40     270           0.129       0.012516
2026-04-24 12:01:01,210 [INFO] ======================================================================
2026-04-24 12:01:01,211 [INFO] Phase 3  |  Day: thursday
2026-04-24 12:01:01,234 [INFO] feature_width=85
2026-04-24 12:01:08,041 [INFO] benign_prefix=251249  fm=8000  ad=40000
2026-04-24 12:01:09,708 [INFO] Fitting HTTPFeatureMapper on 8000 rows …
2026-04-24 12:01:09,712 [INFO] k=17 clusters  sizes=[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
2026-04-24 12:05:07,221 [INFO] Row 200000 / 1174693  836 r/s
2026-04-24 12:08:38,929 [INFO] Row 400000 / 1174693  887 r/s
2026-04-24 

/tmp/ipykernel_55/3684373336.py:424: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipykernel_55/3684373336.py:434: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(prec, rec))


2026-04-24 12:22:20,869 [INFO] [thursday] AUC=0.5534  AUPRC=0.0506  EER=0.4696  F1=0.1033


/tmp/ipykernel_55/3684373336.py:462: RuntimeWarning: invalid value encountered in divide
  f1   = np.where(denom > 0, 2.0 * prec * rec / denom, 0.0)


2026-04-24 12:22:35,974 [INFO] Attack breakdown:
     attack_type  total_rows  attack_rows  detected  missed  detection_rate  threshold_p95
          Benign     1083857            0     54193 1029664          0.0500       0.018512
Brute Force -Web       42329        42329      4716   37613          0.1114       0.018512
   SQL Injection         507          507        48     459          0.0947       0.018512
2026-04-24 12:22:36,108 [INFO] 
2026-04-24 12:22:36,109 [INFO] Phase 3 complete.  Results in: /kaggle/working/results_phase3
2026-04-24 12:22:36,109 [INFO] Summary:
2026-04-24 12:22:36,110 [INFO]   [friday]  AUC=0.5291  AUPRC=0.0621  EER=0.4823  F1=0.1128  k=17
2026-04-24 12:22:36,111 [INFO]   [thursday]  AUC=0.5534  AUPRC=0.0506  EER=0.4696  F1=0.1033  k=17


In [1]:
!zip -r phase3results.zip /kaggle/working/results_phase3 

  adding: kaggle/working/results_phase3/ (stored 0%)
  adding: kaggle/working/results_phase3/summary_metrics.csv (deflated 50%)
  adding: kaggle/working/results_phase3/attack_breakdown_combined.csv (deflated 46%)
  adding: kaggle/working/results_phase3/thursday/ (stored 0%)
  adding: kaggle/working/results_phase3/thursday/roc_curve.csv (deflated 74%)
  adding: kaggle/working/results_phase3/thursday/metrics.json (deflated 54%)
  adding: kaggle/working/results_phase3/thursday/scores.csv (deflated 75%)
  adding: kaggle/working/results_phase3/thursday/pr_curve.csv (deflated 74%)
  adding: kaggle/working/results_phase3/thursday/attack_breakdown.csv (deflated 27%)
  adding: kaggle/working/results_phase3/friday/ (stored 0%)
  adding: kaggle/working/results_phase3/friday/roc_curve.csv (deflated 74%)
  adding: kaggle/working/results_phase3/friday/metrics.json (deflated 53%)
  adding: kaggle/working/results_phase3/friday/scores.csv (deflated 74%)
  adding: kaggle/working/results_phase3/friday/pr